In [1]:
import os 
import matplotlib.pyplot as plt 
import cv2 
import numpy as np
from PIL import Image
from multiprocessing import Pool, cpu_count
import albumentations as A
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from glob import glob
import glob
import shutil
import hashlib
import time

Importing the Dataset

In [2]:
input_path = "/kaggle/input/"
print(os.listdir(input_path))
path = os.path.join(input_path, "cad-cardiac-mri-dataset")

['cad-cardiac-mri-dataset']


Collecting all images in the dataset into Normal and Sick

In [3]:

def collect_images(main_path, category):
    """Recursively collects all image file paths from deeply nested subdirectories."""
    image_files = []  
    for root, _, files in os.walk(main_path):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                image_files.append((os.path.abspath(os.path.join(root, file)), category))  # Store path + category
    return image_files

normal_images = collect_images(os.path.join(path, "Normal"), "Normal")
sick_images = collect_images(os.path.join(path, "Sick"), "Sick")

all_images = normal_images + sick_images

print(f"Unique normal images: {len(normal_images)}")
print(f"Unique sick images: {len(sick_images)}")
print(f"Total collected images: {len(all_images)}")


Unique normal images: 37564
Unique sick images: 25861
Total collected images: 63425


Creating the output directory

In [4]:
TARGET_SIZE = (512, 512) 
TO_GRAYSCALE = True  
APPLY_DENOISING = True  
APPLY_CONTRAST = True  
NORMALIZE_RANGE = (0, 1)  
BATCH_SIZE = 1000 
OUTPUT_DIR = "/kaggle/working/new_dataset"

In [5]:
os.makedirs(os.path.join(OUTPUT_DIR, "Normal"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "Sick"), exist_ok=True)


In [6]:
def preprocess_and_save(image_info):
    img_path, category = image_info 
    try:
        img = Image.open(img_path)
        if TO_GRAYSCALE:
            img = img.convert("L")
        else:
            img = img.convert("RGB")

        img = img.resize(TARGET_SIZE, Image.BICUBIC)
        img = np.array(img)

        if APPLY_DENOISING:
            img = cv2.medianBlur(img, 3)
            img = cv2.GaussianBlur(img, (3, 3), 0)

        if APPLY_CONTRAST:
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
            img = clahe.apply(img) if TO_GRAYSCALE else cv2.cvtColor(clahe.apply(cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)), cv2.COLOR_GRAY2RGB)

        img = img.astype(np.float32) / 255.0
        if NORMALIZE_RANGE == (-1, 1):
            img = img * 2 - 1

        img = (img * 255).astype(np.uint8)

        # Ensure category folder exists
        save_folder = os.path.join(OUTPUT_DIR, category)
        os.makedirs(save_folder, exist_ok=True)

        # Generate a unique filename
        unique_id = hashlib.md5(img_path.encode()).hexdigest()[:8]
        timestamp = int(time.time() * 1000)  # Unique timestamp
        filename = f"{unique_id}_{timestamp}_{os.path.basename(img_path)}"
        save_path = os.path.join(save_folder, filename)

        # Save image
        cv2.imwrite(save_path, img)

    except Exception as e:
        print(f"Error processing {img_path}: {e}")

print("⚡ Preprocessing and saving images...")
for img_info in tqdm(all_images, total=len(all_images)):
    preprocess_and_save(img_info)

print(f"Preprocessing Complete! Images saved in: {OUTPUT_DIR}")

⚡ Preprocessing and saving images...


100%|██████████| 63425/63425 [10:51<00:00, 97.38it/s] 

Preprocessing Complete! Images saved in: /kaggle/working/new_dataset


In [7]:
normal_dir = os.path.join(OUTPUT_DIR, "Normal")
sick_dir = os.path.join(OUTPUT_DIR, "Sick")

num_normal = len(os.listdir(normal_dir)) if os.path.exists(normal_dir) else 0
num_sick = len(os.listdir(sick_dir)) if os.path.exists(sick_dir) else 0

print(f"Normal images: {num_normal}")
print(f"Sick images: {num_sick}")
print(f"Total images: {num_normal + num_sick}")

Normal images: 37564
Sick images: 25861
Total images: 63425


In [ ]:
import random

normal_path = os.path.join(OUTPUT_DIR, "Normal")
sick_path = os.path.join(OUTPUT_DIR, "Sick")

num_images = 10  
cols = 5 
rows = (num_images + cols - 1) // cols  
normal_images = random.sample(os.listdir(normal_path), num_images // 2)
sick_images = random.sample(os.listdir(sick_path), num_images // 2)

image_files = [os.path.join(normal_path, img) for img in normal_images] + \
              [os.path.join(sick_path, img) for img in sick_images]

plt.figure(figsize=(15, rows * 3))

for i in range(len(image_files)):
    img = cv2.imread(image_files[i])

    plt.subplot(rows, cols, i + 1)
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Image {i+1}")

plt.tight_layout()
plt.show()

Sick Data Augmentation for balancing

In [9]:
sick_folder = "/kaggle/working/new_dataset/Sick"

sick_images = glob.glob(os.path.join(sick_folder, "*.*"))
random.shuffle(sick_images)  

augmentations = A.Compose([
    A.HorizontalFlip(),
    A.Rotate(limit=15, p=0.5),
    A.RandomBrightnessContrast(p=1),
    A.GaussianBlur(p=0.3),
    A.RandomResizedCrop(size=(512, 512), scale=(0.8, 1.0), p=0.5)
])

def augment_and_save(image_path, save_dir):
    """Applies augmentation to an image and saves it."""
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    augmented = augmentations(image=img)['image']
    augmented = cv2.cvtColor(augmented, cv2.COLOR_RGB2BGR)
    filename = f"aug_{random.randint(10000, 99999)}_{os.path.basename(image_path)}"
    cv2.imwrite(os.path.join(save_dir, filename), augmented)

# Keep augmenting until Sick folder reaches 37,500 images
print("Augmenting Sick images...")
while len(glob.glob(os.path.join(sick_folder, "*.*"))) < 37500:
    for img_path in tqdm(sick_images, total=len(sick_images)):
        if len(glob.glob(os.path.join(sick_folder, "*.*"))) >= 37500:
            break
        augment_and_save(img_path, sick_folder)

print(f"🎯 Augmentation completed! Total images in Sick: {len(glob.glob(os.path.join(sick_folder, '*.*')))}")


Augmenting Sick images...


 45%|████▌     | 11639/25861 [15:00<18:20, 12.92it/s]


🎯 Augmentation completed! Total images in Sick: 37500


In [10]:
normal_dir = os.path.join(OUTPUT_DIR, "Normal")
sick_dir = os.path.join(OUTPUT_DIR, "Sick")

num_normal = len(os.listdir(normal_dir)) if os.path.exists(normal_dir) else 0
num_sick = len(os.listdir(sick_dir)) if os.path.exists(sick_dir) else 0

print(f"Normal images: {num_normal}")
print(f"Sick images: {num_sick}")
print(f"Total images: {num_normal + num_sick}")


Normal images: 37564
Sick images: 37500
Total images: 75064


Splitting Our dataset

In [11]:
output_path = "/kaggle/working/splitted_dataset"
for split in ["train", "valid", "test"]:
    for category in ["Normal", "Sick"]:
        os.makedirs(os.path.join(output_path, split, category), exist_ok=True)

In [12]:
import os
import shutil
from sklearn.model_selection import train_test_split

# Define input (original dataset) and output (splitted dataset) directories
SOURCE_DIR = "/kaggle/working/new_dataset"  # Change to your actual dataset path
OUTPUT_DIR = "/kaggle/working/splitted_dataset"  # Change to where you want to store the split dataset

# Ensure the output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Define subdirectories inside OUTPUT_DIR for train, val, and test
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(OUTPUT_DIR, split, "Normal"), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, split, "Sick"), exist_ok=True)

# Function to split and copy images
def split_and_copy(class_name):
    class_dir = os.path.join(SOURCE_DIR, class_name)  # Path to original class folder
    images = os.listdir(class_dir)

    # Split dataset (70% Train, 15% Validation, 15% Test)
    train_files, test_val_files = train_test_split(images, test_size=0.3, random_state=42)
    val_files, test_files = train_test_split(test_val_files, test_size=0.5, random_state=42)

    # Define destination directories
    train_dest = os.path.join(OUTPUT_DIR, "train", class_name)
    val_dest = os.path.join(OUTPUT_DIR, "val", class_name)
    test_dest = os.path.join(OUTPUT_DIR, "test", class_name)

    # Copy files to respective directories
    for file in train_files:
        shutil.copy2(os.path.join(class_dir, file), os.path.join(train_dest, file))
    for file in val_files:
        shutil.copy2(os.path.join(class_dir, file), os.path.join(val_dest, file))
    for file in test_files:
        shutil.copy2(os.path.join(class_dir, file), os.path.join(test_dest, file))

    return len(train_files), len(val_files), len(test_files)

# Split and copy images for each class
train_n, val_n, test_n = split_and_copy("Normal")
train_s, val_s, test_s = split_and_copy("Sick")

# Print dataset distribution
print(f"Train: {train_n + train_s} images stored in {os.path.join(OUTPUT_DIR, 'train')}")
print(f"Validation: {val_n + val_s} images stored in {os.path.join(OUTPUT_DIR, 'val')}")
print(f"Test: {test_n + test_s} images stored in {os.path.join(OUTPUT_DIR, 'test')}")

Train: 52544 images stored in /kaggle/working/splitted_dataset/train
Validation: 11260 images stored in /kaggle/working/splitted_dataset/val
Test: 11260 images stored in /kaggle/working/splitted_dataset/test


In [13]:
splitted_dataset = "/kaggle/working/splitted_dataset" 

def count_images(folder):
    return len([f for f in os.listdir(folder) if f.endswith(('.png', '.jpg', '.jpeg'))])

# Define paths
train_normal = os.path.join(splitted_dataset, "train", "Normal")
valid_normal = os.path.join(splitted_dataset, "val", "Normal")
test_normal = os.path.join(splitted_dataset, "test", "Normal")

train_sick = os.path.join(splitted_dataset, "train", "Sick")
valid_sick = os.path.join(splitted_dataset, "val", "Sick")
test_sick = os.path.join(splitted_dataset, "test", "Sick")

# Print counts
print("Training set - Normal:", count_images(train_normal))
print("Validation set - Normal:", count_images(valid_normal))
print("Test set - Normal:", count_images(test_normal))

print("Training set - Sick:", count_images(train_sick))
print("Validation set - Sick:", count_images(valid_sick))
print("Test set - Sick:", count_images(test_sick))


Training set - Normal: 26294
Validation set - Normal: 5635
Test set - Normal: 5635
Training set - Sick: 26250
Validation set - Sick: 5625
Test set - Sick: 5625


In [14]:
train_folder="/kaggle/working/splitted_dataset/train"
test_folder="/kaggle/working/splitted_dataset/test"
validate_folder="/kaggle/working/splitted_dataset/val"

In [15]:
normal_folder_test="/kaggle/working/splitted_dataset/test/Normal"
sick_folder_test="/kaggle/working/splitted_dataset/test/Sick"
normal_folder_train="/kaggle/working/splitted_dataset/train/Normal"
sick_folder_train="/kaggle/working/splitted_dataset/train/Sick"
normal_folder_validate="/kaggle/working/splitted_dataset/val/Normal"
sick_folder_validate="/kaggle/working/splitted_dataset/val/Sick"

In [16]:
import os
import pandas as pd
from tensorflow.keras.preprocessing.image import ImageDataGenerator

def create_dataframe(normal_path, sick_path):
    filepaths = []
    labels = []

    for file in os.listdir(normal_path):
        filepaths.append(os.path.join(normal_path, file))
        labels.append('Normal')
        
    for file in os.listdir(sick_path):
        filepaths.append(os.path.join(sick_path, file))
        labels.append('Sick')
        
    return pd.DataFrame({'Class Path': filepaths, 'Class': labels})

tr_df = create_dataframe(normal_folder_train, sick_folder_train)
valid_df = create_dataframe(normal_folder_validate, sick_folder_validate)
ts_df = create_dataframe(normal_folder_test, sick_folder_test)

print(f"Training Data: {len(tr_df)}")
print(f"Validation Data: {len(valid_df)}")
print(f"Test Data: {len(ts_df)}")

batch_size = 128  # Optimal batch size for speed and memory
img_size = (224, 224)  # 224x224 for VGG16

_gen = ImageDataGenerator(
    rescale=1./255,
    brightness_range=(0.7, 1.3),
    horizontal_flip=True,
    rotation_range=15,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.15
)

ts_gen_conf = ImageDataGenerator(rescale=1./255)


tr_gen = _gen.flow_from_dataframe(
    tr_df, 
    x_col='Class Path',
    y_col='Class', 
    batch_size=batch_size,
    target_size=img_size,
    class_mode='binary' 
)

valid_gen = _gen.flow_from_dataframe(
    valid_df, 
    x_col='Class Path',
    y_col='Class', 
    batch_size=batch_size,
    target_size=img_size,
    class_mode='binary'
)

ts_gen = ts_gen_conf.flow_from_dataframe(
    ts_df, 
    x_col='Class Path',
    y_col='Class', 
    batch_size=16,
    target_size=img_size, 
    shuffle=False,

    class_mode='binary'
)

2025-12-17 09:31:05.092882: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765963865.281927      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765963865.334399      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1765963865.790396      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765963865.790439      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765963865.790445      55 computation_placer.cc:177] computation placer alr

Training Data: 52544
Validation Data: 11260
Test Data: 11260
Found 52544 validated image filenames belonging to 2 classes.
Found 11260 validated image filenames belonging to 2 classes.
Found 11260 validated image filenames belonging to 2 classes.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

normal_samples = ts_df[ts_df['Class'] == 'Normal'].sample(3, random_state=42)
sick_samples = ts_df[ts_df['Class'] == 'Sick'].sample(3, random_state=42)

selection = pd.concat([normal_samples, sick_samples])
plt.figure(figsize=(15, 8))

for i, (index, row) in enumerate(selection.iterrows()):
    img_path = row['Class Path']
    label = row['Class']
    
    img = plt.imread(img_path)
    
    plt.subplot(2, 3, i + 1)
    plt.imshow(img)
    plt.title(label, color='k', fontsize=15)

plt.tight_layout()
plt.show()

In [18]:
import os
from PIL import Image
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob
#---------------------------------------
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
#---------------------------------------
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, BatchNormalization, Input
from tensorflow.keras.optimizers import Adamax
from tensorflow.keras.metrics import Precision, Recall
from tensorflow.keras.preprocessing.image import ImageDataGenerator

#---------------------------------------
import warnings
warnings.filterwarnings("ignore")

# Enable mixed precision for faster training on GPU
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')

print("✅ Mixed precision enabled for faster training")

✅ Mixed precision enabled for faster training


In [ ]:

# 1. Setup Base Model using VGG16
img_shape = (224, 224, 3)  # VGG16 uses 224x224

base_model = tf.keras.applications.VGG16(
    include_top=False,       
    weights="imagenet",      # Using pre-trained ImageNet weights for faster training
    input_shape=img_shape
)

# Freeze base model
base_model.trainable = False

# 2. Build custom classification head with Functional API
inputs = Input(shape=img_shape)
x = base_model(inputs, training=False)  # Frozen layers won't update during training
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
x = BatchNormalization()(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.3)(x)
outputs = Dense(1, activation='sigmoid', dtype='float32')(x)  # Binary classification

model = Model(inputs, outputs, name='VGG16_Cardiac')

# 3. Compile Model
model.compile(
    optimizer=Adamax(learning_rate=0.001),
    
    # binary_crossentropy since output has 1 neuron (binary)
    loss='binary_crossentropy',  
    
    metrics=['accuracy', Precision(), Recall()]
)

model.summary()


I0000 00:00:1765963879.066362      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1765963879.067092      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "DenseNet121_Cardiac"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ densenet121 (Functional)        │ (None, 7, 7, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1024)           │         4,096 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       131,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,181,633 (27.40 MB)

 Trainable params: 141,825 (554.00 KB)

 Non-trainable params: 7,039,808 (26.85 MB)

In [ ]:
tf.keras.utils.plot_model(model, show_shapes=True)

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import json

# Define model name for saving results
MODEL_NAME = "VGG16"
checkpoint_path = f"best_model_{MODEL_NAME}.keras"

callbacks = [
    ModelCheckpoint(
        filepath=checkpoint_path,
        monitor='val_accuracy', 
        mode='max',             
        save_best_only=True,
        verbose=1
    ),
    
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.4, 
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]
epochs = 25

history = model.fit(
    tr_gen,
    validation_data=valid_gen,
    epochs=epochs,
    callbacks=callbacks,
    verbose=1
)

# Save training history to JSON file
history_dict = history.history
# Convert numpy values to Python native types for JSON serialization
for key in history_dict:
    history_dict[key] = [float(val) for val in history_dict[key]]

with open(f'training_history_{MODEL_NAME}.json', 'w') as f:
    json.dump(history_dict, f, indent=4)

print(f"\n✅ Training history saved to: training_history_{MODEL_NAME}.json")


Epoch 1/25


I0000 00:00:1765963902.585738     131 service.cc:152] XLA service 0x79f00c0a1230 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1765963902.585788     131 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1765963902.585794     131 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1765963906.337793     131 cuda_dnn.cc:529] Loaded cuDNN version 91002


  1/411 ━━━━━━━━━━━━━━━━━━━━ 4:39:03 41s/step - accuracy: 0.4375 - loss: 0.8776 - precision: 0.2917 - recall: 0.1129

I0000 00:00:1765963927.368829     131 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


411/411 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6831 - loss: 0.6046 - precision: 0.7040 - recall: 0.6148
Epoch 1: val_accuracy improved from -inf to 0.82789, saving model to best_model_DenseNet121.keras
411/411 ━━━━━━━━━━━━━━━━━━━━ 892s 2s/step - accuracy: 0.6833 - loss: 0.6044 - precision: 0.7041 - recall: 0.6150 - val_accuracy: 0.8279 - val_loss: 0.3762 - val_precision: 0.8045 - val_recall: 0.8660 - learning_rate: 0.0010
Epoch 2/25
411/411 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8057 - loss: 0.4145 - precision: 0.8050 - recall: 0.8080
Epoch 2: val_accuracy improved from 0.82789 to 0.86892, saving model to best_model_DenseNet121.keras
411/411 ━━━━━━━━━━━━━━━━━━━━ 803s 2s/step - accuracy: 0.8057 - loss: 0.4145 - precision: 0.8051 - recall: 0.8080 - val_accuracy: 0.8689 - val_loss: 0.3040 - val_precision: 0.8577 - val_recall: 0.8843 - learning_rate: 0.0010
Epoch 3/25
411/411 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8374 - loss: 0.3555 - precision: 0.8378 - recall: 0.836

In [ ]:
def plot_training_history(history):

    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    
    epochs_range = range(1, len(acc) + 1)

    plt.figure(figsize=(15, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label='Training Accuracy')
    plt.plot(epochs_range, val_acc, label='Validation Accuracy')
    plt.legend(loc='lower right')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')

    # Plot Loss
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Training Loss')
    plt.plot(epochs_range, val_loss, label='Validation Loss')
    plt.legend(loc='upper right')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')

    plt.show()

plot_training_history(history)

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
import json

ts_gen.reset()

print("\nSedang melakukan prediksi...")
predictions = model.predict(ts_gen, verbose=1)

y_pred = (predictions > 0.5).astype(int).ravel()
y_pred_proba = predictions.ravel()  # Keep probabilities for ROC curve

y_true = ts_gen.classes

class_names = list(ts_gen.class_indices.keys()) 

# ============================================
# PERFORMANCE METRICS
# ============================================
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
roc_auc = roc_auc_score(y_true, y_pred_proba)

print("\n" + "="*60)
print(f"📊 PERFORMANCE METRICS - {MODEL_NAME}")
print("="*60)
print(f"Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print("="*60)

# ============================================
# CLASSIFICATION REPORT
# ============================================
print("\n📋 CLASSIFICATION REPORT:")
print("-"*60)
report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
print(classification_report(y_true, y_pred, target_names=class_names))

# ============================================
# CONFUSION MATRIX
# ============================================
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=class_names,
            yticklabels=class_names)
plt.xlabel('Prediksi Model', fontsize=12)
plt.ylabel('Label Asli (Aktual)', fontsize=12)
plt.title(f'Confusion Matrix - {MODEL_NAME}', fontsize=15)
plt.savefig(f'confusion_matrix_{MODEL_NAME}.png', dpi=300, bbox_inches='tight')
plt.show()

# ============================================
# ROC CURVE
# ============================================
fpr, tpr, thresholds = roc_curve(y_true, y_pred_proba)
roc_auc_value = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc_value:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title(f'ROC Curve - {MODEL_NAME}', fontsize=15)
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.savefig(f'roc_curve_{MODEL_NAME}.png', dpi=300, bbox_inches='tight')
plt.show()

# ============================================
# SAVE ALL METRICS
# ============================================
evaluation_results = {
    'model_name': MODEL_NAME,
    'performance_metrics': {
        'accuracy': float(accuracy),
        'precision': float(precision),
        'recall': float(recall),
        'f1_score': float(f1),
        'roc_auc': float(roc_auc)
    },
    'classification_report': report,
    'confusion_matrix': cm.tolist(),
    'total_test_samples': len(y_true),
    'macro_avg': report['macro avg'],
    'weighted_avg': report['weighted avg']
}

with open(f'evaluation_results_{MODEL_NAME}.json', 'w') as f:
    json.dump(evaluation_results, f, indent=4)

print(f"\n✅ Evaluation results saved to: evaluation_results_{MODEL_NAME}.json")
print(f"✅ Confusion matrix saved to: confusion_matrix_{MODEL_NAME}.png")
print(f"✅ ROC curve saved to: roc_curve_{MODEL_NAME}.png")


In [ ]:
# Save final model
model_filename = f"CardiacCADMRI_{MODEL_NAME}_Model.h5"
model.save(model_filename)

print(f"✅ Model berhasil disimpan sebagai: {model_filename}")

# Create a summary report
summary = {
    'model_name': MODEL_NAME,
    'model_architecture': 'VGG16',
    'input_shape': (224, 224, 3),
    'total_params': model.count_params(),
    'optimizer': 'Adamax',
    'learning_rate': 0.001,
    'loss_function': 'binary_crossentropy',
    'epochs_trained': len(history.history['loss']),
    'final_train_accuracy': float(history.history['accuracy'][-1]),
    'final_val_accuracy': float(history.history['val_accuracy'][-1]),
    'final_train_loss': float(history.history['loss'][-1]),
    'final_val_loss': float(history.history['val_loss'][-1]),
    'model_file': model_filename,
    'checkpoint_file': checkpoint_path,
    'training_history_file': f'training_history_{MODEL_NAME}.json',
    'evaluation_results_file': f'evaluation_results_{MODEL_NAME}.json',
    'confusion_matrix_file': f'confusion_matrix_{MODEL_NAME}.png',
    'roc_curve_file': f'roc_curve_{MODEL_NAME}.png'
}

with open(f'model_summary_{MODEL_NAME}.json', 'w') as f:
    json.dump(summary, f, indent=4)

print(f"✅ Model summary saved to: model_summary_{MODEL_NAME}.json")
print("\n" + "="*60)
print("📊 ALL RESULTS SAVED:")
print("="*60)
print(f"1. Model weights: {model_filename}")
print(f"2. Best checkpoint: {checkpoint_path}")
print(f"3. Training history: training_history_{MODEL_NAME}.json")
print(f"4. Evaluation results: evaluation_results_{MODEL_NAME}.json")
print(f"5. Confusion matrix: confusion_matrix_{MODEL_NAME}.png")
print(f"6. ROC curve: roc_curve_{MODEL_NAME}.png")
print(f"7. Model summary: model_summary_{MODEL_NAME}.json")
print("="*60)


✅ Model berhasil disimpan sebagai: CardiacCADMRI_DenseNet121_Model.h5
✅ Model summary saved to: model_summary_DenseNet121.json

📊 ALL RESULTS SAVED:
1. Model weights: CardiacCADMRI_DenseNet121_Model.h5
2. Best checkpoint: best_model_DenseNet121.keras
3. Training history: training_history_DenseNet121.json
4. Evaluation results: evaluation_results_DenseNet121.json
5. Confusion matrix: confusion_matrix_DenseNet121.png
6. ROC curve: roc_curve_DenseNet121.png
7. Model summary: model_summary_DenseNet121.json


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from tensorflow.keras.models import Model
import tensorflow as tf

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    """
    Generate Grad-CAM heatmap for a given image.
    
    Args:
        img_array: Input image array (224, 224, 3)
        model: Trained model
        last_conv_layer_name: Name of the last convolutional layer
        pred_index: Class index for which to compute gradients (None = predicted class)
    
    Returns:
        Heatmap as numpy array
    """
    # Access the base VGG16 model (second layer in your model)
    base_model = model.layers[1]
    last_conv_layer = base_model.get_layer(last_conv_layer_name)
    
    # Create a model that outputs the conv layer activations
    grad_model = Model(
        inputs=[base_model.inputs],
        outputs=[last_conv_layer.output, base_model.output]
    )
    
    # Compute the gradient of the predicted class with respect to the output feature map
    with tf.GradientTape() as tape:
        # Get base model output (through conv layer)
        conv_outputs, base_output = grad_model(img_array)
        
        # Pass through the rest of the model layers
        x = base_output
        for layer in model.layers[2:]:  # Skip input and base_model layers
            x = layer(x)
        preds = x
        
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]
    
    # Gradient of the output neuron with respect to the output feature map
    grads = tape.gradient(class_channel, conv_outputs)
    
    # Vector of mean intensity of the gradient over a specific feature map channel
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    
    # Multiply each channel by "how important it is"
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    
    # Normalize the heatmap between 0 and 1
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-10)
    return heatmap.numpy()


def apply_gradcam_to_image(img_path, model, last_conv_layer_name, img_size=(224, 224), 
                           alpha=0.4, colormap=cv2.COLORMAP_JET):
    """
    Apply Grad-CAM to an image and return the superimposed result.
    
    Args:
        img_path: Path to the image file
        model: Trained model
        last_conv_layer_name: Name of the last convolutional layer
        img_size: Target image size for model input
        alpha: Transparency of heatmap overlay
        colormap: OpenCV colormap for heatmap visualization
    
    Returns:
        original_img, heatmap, superimposed_img, prediction, confidence
    """
    # Load and preprocess image
    img = tf.keras.preprocessing.image.load_img(img_path, target_size=img_size)
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = img_array / 255.0
    
    # Get model prediction
    preds = model.predict(img_array, verbose=0)
    predicted_class = int(preds[0] > 0.5)
    confidence = float(preds[0][0]) if predicted_class == 1 else float(1 - preds[0][0])
    
    # Generate Grad-CAM heatmap
    heatmap = make_gradcam_heatmap(img_array, model, last_conv_layer_name)
    
    # Ensure heatmap is valid and properly shaped
    if np.isnan(heatmap).any() or np.isinf(heatmap).any():
        heatmap = np.nan_to_num(heatmap, nan=0.0, posinf=1.0, neginf=0.0)
    
    # Ensure heatmap is 2D and has proper dtype
    if len(heatmap.shape) != 2:
        raise ValueError(f"Heatmap must be 2D, got shape {heatmap.shape}")
    
    heatmap = heatmap.astype(np.float32)
    
    # Resize heatmap to match original image size
    heatmap = cv2.resize(heatmap, (img_size[0], img_size[1]), interpolation=cv2.INTER_LINEAR)
    
    # Convert heatmap to RGB
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, colormap)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    
    # Superimpose heatmap on original image
    original_img = np.uint8(img_array[0] * 255)
    superimposed_img = cv2.addWeighted(original_img, 1 - alpha, heatmap, alpha, 0)
    
    return original_img, heatmap, superimposed_img, predicted_class, confidence


print("✅ Grad-CAM functions defined successfully!")

✅ Grad-CAM functions defined successfully!


In [ ]:
# Find the last convolutional layer in VGG16
# Access the base VGG16 model (it's the second layer: index 1)
base_model = model.layers[1]

# For VGG16, the last convolutional layer is 'block5_conv3'
# Let's verify it exists and find alternatives if needed
last_conv_layer_name = None

for layer in reversed(base_model.layers):
    if 'conv' in layer.name.lower() or 'block' in layer.name.lower():
        # Check if this layer has output (not all layers do)
        try:
            _ = layer.output
            last_conv_layer_name = layer.name
            break
        except:
            continue

if last_conv_layer_name is None:
    # Fallback to the commonly known last layer for VGG16
    last_conv_layer_name = 'block5_conv3'

print(f"🔍 Using last convolutional layer: {last_conv_layer_name}")
print(f"📦 Base model layers: {len(base_model.layers)}")

🔍 Using last convolutional layer: conv5_block16_concat
📦 Base model layers: 427


In [ ]:
# Visualize Grad-CAM on sample test images
# Select random samples from each class
num_samples = 3  # Number of samples per class

normal_samples = ts_df[ts_df['Class'] == 'Normal'].sample(num_samples, random_state=42)
sick_samples = ts_df[ts_df['Class'] == 'Sick'].sample(num_samples, random_state=42)

# Combine samples
sample_images = pd.concat([normal_samples, sick_samples])

# Get class names
class_names = list(ts_gen.class_indices.keys())

# Create visualization
fig = plt.figure(figsize=(18, num_samples * 2 * 3))

for idx, (_, row) in enumerate(sample_images.iterrows()):
    img_path = row['Class Path']
    true_label = row['Class']
    
    # Apply Grad-CAM
    original, heatmap, superimposed, pred_class, confidence = apply_gradcam_to_image(
        img_path, model, last_conv_layer_name
    )
    
    pred_label = class_names[pred_class]
    
    # Plot original image
    plt.subplot(num_samples * 2, 3, idx * 3 + 1)
    plt.imshow(original)
    plt.title(f"Original\nTrue: {true_label}", fontsize=10)
    plt.axis('off')
    
    # Plot heatmap
    plt.subplot(num_samples * 2, 3, idx * 3 + 2)
    plt.imshow(heatmap)
    plt.title(f"Grad-CAM Heatmap", fontsize=10)
    plt.axis('off')
    
    # Plot superimposed
    plt.subplot(num_samples * 2, 3, idx * 3 + 3)
    plt.imshow(superimposed)
    color = 'green' if pred_label == true_label else 'red'
    plt.title(f"Overlay\nPred: {pred_label} ({confidence:.2%})", fontsize=10, color=color)
    plt.axis('off')

plt.tight_layout()
plt.savefig(f'gradcam_visualization_{MODEL_NAME}.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ Grad-CAM visualization saved to: gradcam_visualization_{MODEL_NAME}.png")

In [ ]:
# Create a detailed Grad-CAM comparison for correctly vs incorrectly classified images

# Get predictions for all test images
ts_gen.reset()
all_preds = model.predict(ts_gen, verbose=1)
all_pred_classes = (all_preds > 0.5).astype(int).ravel()
all_true_classes = ts_gen.classes

# Find correctly and incorrectly classified images
correct_indices = np.where(all_pred_classes == all_true_classes)[0]
incorrect_indices = np.where(all_pred_classes != all_true_classes)[0]

print(f"Correctly classified: {len(correct_indices)}")
print(f"Incorrectly classified: {len(incorrect_indices)}")

# Sample some correctly and incorrectly classified images
num_correct_samples = min(3, len(correct_indices))
num_incorrect_samples = min(3, len(incorrect_indices))

if len(correct_indices) > 0:
    selected_correct = np.random.choice(correct_indices, num_correct_samples, replace=False)
else:
    selected_correct = []

if len(incorrect_indices) > 0:
    selected_incorrect = np.random.choice(incorrect_indices, num_incorrect_samples, replace=False)
else:
    selected_incorrect = []

# Visualize
fig = plt.figure(figsize=(18, 10))

# Plot correctly classified
for i, idx in enumerate(selected_correct):
    img_path = ts_df.iloc[idx]['Class Path']
    true_label = ts_df.iloc[idx]['Class']
    
    original, heatmap, superimposed, pred_class, confidence = apply_gradcam_to_image(
        img_path, model, last_conv_layer_name
    )
    
    pred_label = class_names[pred_class]
    
    plt.subplot(2, num_correct_samples, i + 1)
    plt.imshow(superimposed)
    plt.title(f"✓ CORRECT\nTrue: {true_label}\nPred: {pred_label} ({confidence:.2%})", 
              fontsize=10, color='green', weight='bold')
    plt.axis('off')

# Plot incorrectly classified
for i, idx in enumerate(selected_incorrect):
    img_path = ts_df.iloc[idx]['Class Path']
    true_label = ts_df.iloc[idx]['Class']
    
    original, heatmap, superimposed, pred_class, confidence = apply_gradcam_to_image(
        img_path, model, last_conv_layer_name
    )
    
    pred_label = class_names[pred_class]
    
    plt.subplot(2, num_correct_samples, num_correct_samples + i + 1)
    plt.imshow(superimposed)
    plt.title(f"✗ INCORRECT\nTrue: {true_label}\nPred: {pred_label} ({confidence:.2%})", 
              fontsize=10, color='red', weight='bold')
    plt.axis('off')

plt.tight_layout()
plt.savefig(f'gradcam_correct_vs_incorrect_{MODEL_NAME}.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ Comparison visualization saved to: gradcam_correct_vs_incorrect_{MODEL_NAME}.png")